# K-Means Clustering

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/clustering/01-k-means

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — group by nearest center

**K-Means** partitions data into `K` clusters by alternating two simple steps: **assign** each point to
its nearest **centroid**, then **update** each centroid to the mean of its assigned points. Repeat
until nothing moves. It's minimizing the total **within-cluster squared distance** (inertia), and the
"update = mean" step is exactly what minimizes that objective. The catches: you must **choose K** (the
elbow method and silhouette score help), it assumes **spherical, similar-size** clusters, and it's
sensitive to **initialization** (hence k-means++ and multiple restarts). We build it from scratch and
validate against `sklearn`.

## K-Means Algorithm

1. Initialize centroids randomly
2. Assign each point to nearest centroid
3. Update centroids as mean of assigned points
4. Repeat until convergence

In [ ]:
np.random.seed(42)
K = 3
centers = np.array([[0, 0], [5, 0], [2.5, 4.3]])
n_per = 50
X = np.vstack([np.random.randn(n_per, 2) * 0.8 + c for c in centers])

def kmeans(X, K, n_iter=10):
    centroids = X[np.random.choice(len(X), K, replace=False)]
    history = [centroids.copy()]
    for _ in range(n_iter):
        dists = np.linalg.norm(X[:, None] - centroids[None], axis=2)
        labels = np.argmin(dists, axis=1)
        centroids = np.array([X[labels == k].mean(axis=0) for k in range(K)])
        history.append(centroids.copy())
    return labels, centroids, history

labels, centroids, history = kmeans(X, K)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, ax in enumerate(axes):
    step_centroids = history[i]
    dists = np.linalg.norm(X[:, None] - step_centroids[None], axis=2)
    step_labels = np.argmin(dists, axis=1)
    for k in range(K):
        mask = step_labels == k
        ax.scatter(X[mask, 0], X[mask, 1], c=['#818cf8', '#14b8a6', '#eab308'][k], s=10, alpha=0.5)
    ax.scatter(step_centroids[:, 0], step_centroids[:, 1], c='white', s=200, marker='*', edgecolors='black', linewidths=1)
    ax.set_title(f'Iteration {i}', color='white', fontsize=11)
    ax.set_xlim(-2, 7)
    ax.set_ylim(-2, 6)
    ax.set_aspect('equal')
    ax.axis('off')
plt.suptitle('K-Means Convergence', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**What to notice:** across the four iterations the centroids (white stars) march from their random
start to the three true cluster centers, and the point colorings snap into clean groups. Each
iteration strictly reduces the inertia, so k-means always **converges** — though only to a *local*
optimum (a gotcha below).

## The library way — validate against `sklearn`

`sklearn.cluster.KMeans` runs the same assign/update loop (with k-means++ init and restarts). Since
cluster *labels* are arbitrary (cluster 0 vs 1 is just a name), we compare partitions with the
**adjusted Rand index** (1.0 = identical grouping).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

sk = KMeans(n_clusters=K, n_init=10, random_state=0).fit(X)
ari = adjusted_rand_score(labels, sk.labels_)
print(f'adjusted Rand index (our k-means vs sklearn): {ari:.3f}')
assert ari > 0.95, "our clustering must match sklearn's grouping"
print('our from-scratch k-means == sklearn KMeans (same partition) ✓')

**What to notice:** an ARI of ~1.0 — our from-scratch k-means recovers the *same* clustering as
`sklearn`, up to the arbitrary naming of clusters. On well-separated blobs the assign/update loop is
robust; the difficulty is all in choosing `K` and handling awkward cluster shapes.

## Why the update is the mean — and a worked example

With assignments fixed, cluster $k$ only appears in $\sum_{x_i \in C_k}\lVert x_i-\mu_k\rVert^2$. Setting the gradient to zero,

$$\frac{\partial}{\partial\mu_k}\sum_{x_i\in C_k}\lVert x_i-\mu_k\rVert^2=\sum_{x_i\in C_k}-2(x_i-\mu_k)=0\;\Rightarrow\;\mu_k=\frac{1}{|C_k|}\sum_{x_i\in C_k}x_i.$$

So the **mean minimizes within-cluster squared distance** — that is *why* we move each centroid to its cluster mean. The assign step lowers $J$ (each point picks its nearest centroid) and the update step lowers $J$ (mean is optimal), so $J$ decreases monotonically. Below we verify both on the 5-point example from the lesson.

In [ ]:
import numpy as np

# 5-point worked example (A..E), K=2, init mu1=A, mu2=E
pts = {'A': (1, 1), 'B': (1.5, 2), 'C': (3, 4), 'D': (5, 7), 'E': (3.5, 5)}
P = np.array(list(pts.values())); names = list(pts.keys())
mu = np.array([[1.0, 1.0], [3.5, 5.0]])   # mu1 = A, mu2 = E

def sq(a, b):
    return float(np.sum((np.array(a) - np.array(b))**2))

def assign(P, mu):
    d = np.array([[sq(p, m) for m in mu] for p in P])
    return d, d.argmin(axis=1)

def inertia(P, labels, mu):
    return sum(sq(P[i], mu[labels[i]]) for i in range(len(P)))

# --- Iteration 1: assign (squared-distance table) ---
d, labels = assign(P, mu)
print('Iter 1 assign  | d^2 to mu1   d^2 to mu2  -> cluster')
for i, n in enumerate(names):
    print(f'  {n}{tuple(P[i])}  |   {d[i,0]:6.2f}      {d[i,1]:6.2f}    ->  C{labels[i]+1}')
J_before = inertia(P, labels, mu)
print(f'\nJ with OLD centroids (post-assign) = {J_before:.2f}')

# --- Iteration 1: update (mean of each cluster) ---
mu_new = np.array([P[labels == k].mean(axis=0) for k in range(2)])
J_after = inertia(P, labels, mu_new)
print(f'New centroids: mu1={tuple(np.round(mu_new[0],3))}, mu2={tuple(np.round(mu_new[1],3))}')
print(f'J with NEW centroids               = {J_after:.2f}   ({J_after:.2f} < {J_before:.2f}  ✓ update lowered J)')

# Verify the mean BEATS any nearby centroid choice (mean is the minimizer)
c0 = P[labels == 1]                      # cluster C2 points
grid = c0.mean(0) + np.random.default_rng(0).normal(0, 0.3, size=(2000, 2))
best = min(sum(sq(p, g) for p in c0) for g in grid)
mean_sse = sum(sq(p, c0.mean(0)) for p in c0)
print(f'\nMean SSE for C2 = {mean_sse:.3f};  best of 2000 random nearby centroids = {best:.3f}'
      f'  -> mean is optimal: {mean_sse <= best}')

# --- Iteration 2: reassign with new centroids -> should be unchanged (converged) ---
_, labels2 = assign(P, mu_new)
print(f'\nIter 2 labels {list(labels2)}  ==  Iter 1 labels {list(labels)}  -> converged: {np.array_equal(labels, labels2)}')
print('Final: C1 =', [names[i] for i in range(5) if labels[i]==0],
      ' C2 =', [names[i] for i in range(5) if labels[i]==1])


**What to notice:** the worked example shows *why* the update step uses the **mean** — the mean is
the point minimizing the sum of squared distances to a set, so setting each centroid to its cluster's
mean is the optimal update for the inertia objective. K-means is coordinate descent on that objective.

## Elbow Method

Plot inertia vs K — look for the "elbow" where improvement slows.

In [ ]:
inertias = []
for k in range(1, 10):
    _, cents, _ = kmeans(X, k)
    dists = np.linalg.norm(X[:, None] - cents[None], axis=2)
    labels = np.argmin(dists, axis=1)
    inertia = sum(np.sum((X[labels == k] - cents[k])**2) for k in range(k))
    inertias.append(inertia)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(range(1, 10), inertias, 'o-', color='#818cf8', linewidth=2, markersize=8)
axes[0].axvline(3, color='#f43f5e', linestyle='--', alpha=0.7, label='Elbow at K=3')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia (Within-Cluster SS)')
axes[0].set_title('Elbow Method', color='white')
axes[0].legend()

silhouette_scores = []
for k in range(2, 10):
    _, cents, _ = kmeans(X, k)
    dists = np.linalg.norm(X[:, None] - cents[None], axis=2)
    labels = np.argmin(dists, axis=1)
    s = 0
    for i in range(len(X)):
        ci = labels[i]
        a = np.mean(np.linalg.norm(X[labels == ci] - X[i], axis=1))
        b = min(np.mean(np.linalg.norm(X[labels == j] - X[i], axis=1)) for j in range(k) if j != ci)
        s += (b - a) / max(a, b)
    silhouette_scores.append(s / len(X))
axes[1].plot(range(2, 10), silhouette_scores, 's-', color='#14b8a6', linewidth=2, markersize=8)
axes[1].axvline(3, color='#f43f5e', linestyle='--', alpha=0.7, label='Best at K=3')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score', color='white')
axes[1].legend()
plt.tight_layout()
plt.show()

**What to notice:** the **elbow method** plots inertia vs `K` — it drops steeply while you're
splitting real clusters, then flattens once you're just subdividing them. The "elbow" (the bend) marks
a good `K`. It's a heuristic, not a proof — the bend can be ambiguous, which is why silhouette is a
useful second opinion.

## Silhouette score: validating K

The silhouette compares each point's cohesion (own cluster) to its separation (nearest other cluster). Values near 1 are good; the best $K$ maximizes the mean.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=400, centers=4, cluster_std=0.8, random_state=0)
for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    print(f'K={k}: silhouette = {silhouette_score(X, labels):.3f}')

**What to notice:** the **silhouette score** (how much closer each point is to its own cluster than
the next-nearest) peaks at the **true K=4** for this data. Unlike the elbow, it gives a single number
per `K` to compare, and needs no elbow-eyeballing. Together the two methods triangulate the right
number of clusters.

## Gotchas & tradeoffs

- **You must choose K.** K-means can't discover the number of clusters — use elbow/silhouette, or a
  method like DBSCAN that finds it automatically.
- **It assumes spherical, similar-size clusters.** On elongated, non-convex, or unequal-density
  clusters it fails badly (next lesson: DBSCAN handles these).
- **Initialization matters.** Random starts can land in bad local optima; **k-means++** and multiple
  restarts (`n_init`) mitigate it.
- **Scale-sensitive and outlier-sensitive.** Distances dominate, so standardize features, and remember
  the mean is pulled by outliers.

In [ ]:
# K-means assumes spherical clusters -> it fails on non-convex shapes (two interleaved moons)
from sklearn.datasets import make_moons
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

Xm, ym = make_moons(n_samples=300, noise=0.06, random_state=0)
km_labels = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(Xm)
print(f'k-means ARI on two moons: {adjusted_rand_score(ym, km_labels):.3f}  (near 0 -> wrong clustering)')
print('-> k-means splits the moons with a straight line; it cannot follow the crescents (use DBSCAN)')

**What to notice:** on two interleaved crescents k-means scores a near-zero ARI — it can only carve
**convex, roughly spherical** regions, so it slices each moon in half instead of following its shape.
Density-based clustering (DBSCAN, next lesson) handles exactly these cases. Know your algorithm's
assumptions before trusting its clusters.

## Key takeaways

- K-Means alternates **assign to nearest centroid** and **move centroid to mean** until stable.
- It minimizes within-cluster variance (inertia) but assumes spherical, similar-size clusters.
- Pick $K$ with the **elbow** (inertia) or **silhouette** score; use **K-Means++** init and `n_init>1`.
- Always **standardize** features — K-Means uses Euclidean distance.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — The assign step

Half of K-Means: send every point to its **nearest centroid**. Implement it vectorized — compute all point-to-centroid distances, then `argmin` across centroids.

In [ ]:
def assign_clusters(X, centroids):
    """Label each point with the index of its nearest centroid."""
    X = np.asarray(X, dtype=float)
    centroids = np.asarray(centroids, dtype=float)

    # TODO(you): distances of every point to every centroid
    # (hint: np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2))
    dists = ...

    # TODO(you): index of the nearest centroid per point (hint: np.argmin with axis=1)
    return ...

In [ ]:
# Checks — run me
X = np.array([[0.0, 0.0], [1.0, 0.0], [10.0, 10.0], [11.0, 10.0]])
C = np.array([[0.5, 0.0], [10.5, 10.0]])
assert list(assign_clusters(X, C)) == [0, 0, 1, 1], "each point joins its nearest centroid"
assert list(assign_clusters([[4.9, 0.0]], np.array([[0.0, 0.0], [10.0, 0.0]]))) == [0], \
    "4.9 is closer to 0 than to 10"

# Edge case: k > number of data points -- assignment must still work fine;
# some centroids simply end up with zero points (Exercise 2 handles that part).
X_small = np.array([[0.0, 0.0], [1.0, 0.0]])
C_many = np.array([[0.0, 0.0], [1.0, 0.0], [50.0, 50.0]])   # k=3, only 2 points
assert list(assign_clusters(X_small, C_many)) == [0, 1], "each point still joins its own nearest centroid"

# Edge case: all-identical input points -- every point ties for distance to
# two symmetric centroids; argmin must break the tie the same way for all of them.
X_dup = np.array([[0.0, 0.0]] * 4)
sym_C = np.array([[-1.0, 0.0], [1.0, 0.0]])
assert list(assign_clusters(X_dup, sym_C)) == [0, 0, 0, 0], \
    "identical points must all break the tie toward the same (lower-index) centroid"

print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def assign_clusters(X, centroids):
    X = np.asarray(X, dtype=float)
    centroids = np.asarray(centroids, dtype=float)
    dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
    return np.argmin(dists, axis=1)
```

</details>

### Exercise 2 — The update step, empty clusters, and convergence

The other half: each centroid moves to the **mean** of the points assigned to it — the mean is exactly the point that minimizes within-cluster squared distance, which is why K-Means converges. But what happens if a centroid loses **every** point during an iteration (an *empty cluster*)? This is a real, common edge case — it happens whenever `k` exceeds the number of distinct point locations, or a bad initialization strands a centroid too far from everything. There's no mean of zero points, so instead of producing `NaN`, keep that centroid exactly where it was. The checks below cover: a normal update, the fixed point where assigning and updating change nothing (convergence), and the empty-cluster fallback itself.

In [ ]:
def update_centroids(X, labels, k, prev_centroids):
    """New centroid j = mean of the points with label j.

    Edge case: if no points are assigned to cluster j (an *empty cluster*),
    there's no mean to take -- fall back to that cluster's previous centroid
    instead of producing NaN.
    """
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)
    prev_centroids = np.asarray(prev_centroids, dtype=float)

    new_centroids = []
    for j in range(k):
        mask = labels == j
        # TODO(you): mean of X[mask] if cluster j has any points assigned
        # (hint: np.any(mask)), otherwise prev_centroids[j]
        new_centroids.append(...)
    return np.array(new_centroids)

In [ ]:
# Checks — run me
X = np.array([[0.0, 0.0], [1.0, 0.0], [10.0, 10.0], [11.0, 10.0]])
old_C = np.array([[0.0, 0.0], [10.0, 10.0]])
new_C = update_centroids(X, np.array([0, 0, 1, 1]), 2, old_C)
assert np.allclose(new_C, [[0.5, 0.0], [10.5, 10.0]]), "each non-empty centroid moves to its cluster's mean"

labels = assign_clusters(X, new_C)
assert np.allclose(update_centroids(X, labels, 2, new_C), new_C), \
    "assign + update changes nothing: K-Means has converged"

# Edge case: empty cluster -- nobody assigned to cluster 1. Must fall back to
# its previous centroid, not NaN, while cluster 0 still updates normally.
labels_empty = np.array([0, 0, 0, 0])
prev = np.array([[0.0, 0.0], [99.0, 99.0]])
result = update_centroids(X, labels_empty, 2, prev)
assert not np.isnan(result).any(), "empty cluster must not produce NaN"
assert np.allclose(result[1], prev[1]), "empty cluster keeps its previous centroid"
assert np.allclose(result[0], X.mean(axis=0)), "non-empty cluster still updates to the mean"

# Edge case: k > number of data points -- the "extra" centroid is necessarily empty.
X_small = np.array([[0.0, 0.0], [1.0, 0.0]])
prev_small = np.array([[0.0, 0.0], [1.0, 0.0], [50.0, 50.0]])
labels_small = assign_clusters(X_small, prev_small)   # k=3 centroids, only 2 points
result_small = update_centroids(X_small, labels_small, 3, prev_small)
assert not np.isnan(result_small).any(), "k > n_points must not produce NaN anywhere"
assert np.allclose(result_small[2], prev_small[2]), "the unused (3rd) centroid stays put"

# Edge case: all-identical input points (degenerate cluster) -- ties break every
# point to the same centroid, so the other centroid ends up with zero points.
X_dup = np.array([[0.0, 0.0]] * 4)
sym_C = np.array([[-1.0, 0.0], [1.0, 0.0]])
labels_dup = assign_clusters(X_dup, sym_C)
result_dup = update_centroids(X_dup, labels_dup, 2, sym_C)
assert np.allclose(result_dup[0], [0.0, 0.0]), "occupied cluster's mean is the shared point"
assert np.allclose(result_dup[1], sym_C[1]), "the empty cluster keeps its previous centroid"

print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def update_centroids(X, labels, k, prev_centroids):
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)
    prev_centroids = np.asarray(prev_centroids, dtype=float)
    new_centroids = []
    for j in range(k):
        mask = labels == j
        new_centroids.append(X[mask].mean(axis=0) if np.any(mask) else prev_centroids[j])
    return np.array(new_centroids)
```

</details>

---
## 🔬 Extra practice — DML `17_k-means-clustering`

[Open-Deep-ML problem 17](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/17_k-means-clustering) asks for the whole assign/update loop as a single function with a fixed `max_iterations` cap and an explicit (not random) `initial_centroids` list — closer to what a test harness actually calls. Match its exact signature:

```python
def k_means_clustering(points: list[tuple[float, float]], k: int, initial_centroids: list[tuple[float, float]], max_iterations: int) -> list[tuple[float, float]]:
    ...
```

Each output centroid is **rounded to 4 decimals**, and the loop stops early the moment centroids stop moving. Reuse `assign_clusters` and `update_centroids` from above — this is also a good place to see the empty-cluster fallback fire for real, since DML's own test bank calls this with `k` larger than the number of distinct point locations.

In [ ]:
def k_means_clustering(points, k, initial_centroids, max_iterations):
    """DML 17: run assign/update for up to max_iterations rounds, stopping
    early on convergence. Centroids are rounded to 4 decimals after each
    update, matching DML's expected output format."""
    points = np.asarray(points, dtype=float)
    centroids = np.asarray(initial_centroids, dtype=float)

    for _ in range(max_iterations):
        # TODO(you): assign every point to its nearest centroid, then compute
        # the updated centroids (remember update_centroids needs the *current*
        # centroids as its empty-cluster fallback), then round to 4 decimals.
        labels = ...
        new_centroids = ...
        new_centroids = np.round(new_centroids, 4)

        # TODO(you): stop early once centroids stop moving (converged)
        if ...:
            centroids = new_centroids
            break
        centroids = new_centroids

    return [tuple(c) for c in centroids]

In [ ]:
# Checks — run me (DML's own published test cases, from tests.json)
assert k_means_clustering([(1, 2), (1, 4), (1, 0), (10, 2), (10, 4), (10, 0)], 2,
                           [(1, 1), (10, 1)], 10) == [(1.0, 2.0), (10.0, 2.0)]

assert k_means_clustering([(0, 0, 0), (2, 2, 2), (1, 1, 1), (9, 10, 9), (10, 11, 10), (12, 11, 12)], 2,
                           [(1, 1, 1), (10, 10, 10)], 10) == [(1.0, 1.0, 1.0), (10.3333, 10.6667, 10.3333)]

assert k_means_clustering([(1, 1), (2, 2), (3, 3), (4, 4)], 1, [(0, 0)], 10) == [(2.5, 2.5)]

assert k_means_clustering(
    [(0, 0), (1, 0), (0, 1), (1, 1), (5, 5), (6, 5), (5, 6), (6, 6),
     (0, 5), (1, 5), (0, 6), (1, 6), (5, 0), (6, 0), (5, 1), (6, 1)],
    4, [(0, 0), (0, 5), (5, 0), (5, 5)], 10
) == [(0.5, 0.5), (0.5, 5.5), (5.5, 0.5), (5.5, 5.5)]

# Edge case: k larger than the number of distinct point locations -- one
# centroid is stranded with zero points and must simply stay put.
assert k_means_clustering([(0, 0), (0, 0)], 3, [(0, 0), (5, 5), (-5, -5)], 5) == \
    [(0.0, 0.0), (5.0, 5.0), (-5.0, -5.0)]

print("✅ DML 17 k-means-clustering passed")

<details>
<summary>💡 Show solution</summary>

```python
def k_means_clustering(points, k, initial_centroids, max_iterations):
    points = np.asarray(points, dtype=float)
    centroids = np.asarray(initial_centroids, dtype=float)
    for _ in range(max_iterations):
        labels = assign_clusters(points, centroids)
        new_centroids = update_centroids(points, labels, k, centroids)
        new_centroids = np.round(new_centroids, 4)
        if np.allclose(centroids, new_centroids):
            centroids = new_centroids
            break
        centroids = new_centroids
    return [tuple(c) for c in centroids]
```

</details>